# 01b. Deterministic TEA + LCI reproducibility tables

This notebook is a **reproducibility companion to `01_deterministic_lca_tea.ipynb`**. It does not introduce a second model implementation. Instead, it uses the same clean `graphite_sus` deterministic workflow, exports the exact inputs/accounting terms needed by an external reader, reloads those exported files from disk, and verifies numerical closure.

## Outputs

### TEA
- `tea_msp_breakdown.csv` — exact Figure-3 main cost-category contributions in $/kg and % of MSP.
- `tea_streams.csv` — LCI-compiled physical TEA streams, native units, and normalized quantities per kg graphite.
- `tea_prices.csv` — deterministic prices actually used by each pathway.
- `tea_capex_inputs.csv` — pathway-specific CAPEX input rows.
- `tea_labour_inputs.csv` — labor input rows.
- `tea_financial_assumptions.csv` — named `literature_hybrid` financial/capital assumptions.
- `tea_capital_summary.csv` — calculated capital basis used in the TEA.
- `tea_cashflows.csv` — solved annual cash flows at MSP.
- `tea_validation.csv` — result-level and exported-input round-trip checks.

### LCI / LCIA factor mapping
- `lci_foreground_inventory.csv` — evaluated physical exchanges per 1 unit of the collapsed graphite system activity, independent of a particular LCIA database.
- `lci_allocation_audit.csv` — H2/hydrocarbon coproduct masses and graphite mass-allocation fraction.
- `lcia_factor_mapping_reference.csv` — exact current UF/CF factors used by the paper, for verification only.
- `lcia_factor_mapping_template.csv` — the same mapping identifiers with a blank `user_factor` column so another analyst can map their own unit impact factors/CFs.
- `lci_validation.csv` — 4 pathways × 6 impacts = 24 exact round-trip LCIA checks.

The intended external calculation is

\[
I_{p,k}=\sum_i q_{p,i} F_{i,k},
\qquad
I^{mass}_{p,k}=f^{mass}_{p}\, I_{p,k},
\]

where `q` comes from `lci_foreground_inventory.csv` and `F` comes from the user's mapped unit-impact-factor table. **Rows marked `include_in_lcia=False` are internal materialized links and must not be assigned a second background factor.**

## 0. Paths and deterministic controls

Keep these controls aligned with notebook 01. The notebook is intentionally self-contained and can be run independently after the project databases are available.

In [1]:
from pathlib import Path
import dataclasses
import hashlib
import json
import math
import re
import sys
import warnings

import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
_candidates = [HERE, HERE.parent, HERE.parent.parent]
PROJECT_ROOT = next((p for p in _candidates if (p / "graphite_sus").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate clean project root containing graphite_sus/")
sys.path.insert(0, str(PROJECT_ROOT))

import graphite_sus

PKG_DIR = Path(graphite_sus.__file__).resolve().parent
LCI_XLSX = PKG_DIR / "data" / "test" / "meb_bw_char_coke_v96.xlsx"
TEA_XLSX = PKG_DIR / "data" / "test" / "meb_tea_char_coke_v53.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "result" / "reproducibility"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_NAME = "graphite_sys_v3"
REBUILD_DATABASES = False
TEA_PROFILE = "literature_hybrid"
ELECTRICITY_MODE = "national_2025"
SCENARIOS = ("s_c1", "s_c2", "s_o1", "s_o2")
PATHWAY = {"s_c1":"BC1", "s_c2":"BC2", "s_o1":"BO1", "s_o2":"BO2"}

for p in (LCI_XLSX, TEA_XLSX):
    if not p.exists():
        raise FileNotFoundError(p)

print("Project root:", PROJECT_ROOT)
print("Package:", PKG_DIR)
print("Output:", OUTPUT_DIR)
print("LCI workbook:", LCI_XLSX)
print("TEA workbook:", TEA_XLSX)

Project root: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2
Package: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/graphite_sus
Output: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/reproducibility
LCI workbook: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/graphite_sus/data/test/meb_bw_char_coke_v96.xlsx
TEA workbook: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/graphite_sus/data/test/meb_tea_char_coke_v53.xlsx


## 1. Set/validate Brightway databases and current deterministic factors

This mirrors notebook 01's reference gates. It matters because the collapsed system stores the authoritative `UF_*`/`CF_*` parameters used by the paper, including post-build EPA/NETL fuel-factor overrides.

In [2]:
from graphite_sus.database_setup import ensure_databases
from graphite_sus.fuel_reference import audit_system_fuel_reference, fuel_reference_manifest
from graphite_sus.electricity_regeneration import (
    audit_system_electricity_reference,
    electricity_reference_manifest as electricity_ef_reference_manifest,
)

_ = ensure_databases(
    project_name=PROJECT_NAME,
    lci_workbook=LCI_XLSX,
    rebuild=REBUILD_DATABASES,
)

fuel_ef_audit = audit_system_fuel_reference("graphite_system")
electricity_ef_audit = audit_system_electricity_reference("graphite_system")
assert not fuel_ef_audit.empty and bool(fuel_ef_audit["passed"].all())
assert not electricity_ef_audit.empty and bool(electricity_ef_audit["passed"].all())

fuel_ef_audit.to_csv(OUTPUT_DIR / "fuel_ef_reference_audit.csv", index=False)
electricity_ef_audit.to_csv(OUTPUT_DIR / "electricity_ef_reference_audit.csv", index=False)
print("Fuel-factor reference gate: PASS")
print("Electricity-factor reference gate: PASS")

Fuel-factor reference gate: PASS
Electricity-factor reference gate: PASS


## 2. Build the authoritative deterministic TEA cache

The cache is created by the exact same helper used in notebook 01. This establishes the authoritative MSP, cash flows, physical streams, and Figure-3 contribution accounting against which the exported files will be checked.

In [3]:
from graphite_sus.plots.tea_figure3 import build_tea_figure_cache, validate_figure3_cache

tea_cache = build_tea_figure_cache(
    TEA_XLSX,
    scenarios=SCENARIOS,
    tea_profile=TEA_PROFILE,
    electricity_mode=ELECTRICITY_MODE,
)
tea_authoritative_validation = validate_figure3_cache(tea_cache)
assert bool(tea_authoritative_validation["passes_closure"].all())
assert bool(tea_authoritative_validation["passes_irr"].all())
display(tea_authoritative_validation)

{'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Carbon_dioxide_fossil__air__urban_air_close_to_ground': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char', 'amount': 1.0}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Dinitrogen_monoxide__air__non_urban_air_or_from_high_stacks': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char', 'amount': 273.0}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Dinitrogen_monoxide__air__urban_air_close_to_ground': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char', 'amount': 273.0}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Methane_fossil__air__urban_air_close_to_ground': {'database': 'graphite_system', 'code': 'system::graphite_prod_char::graphtz_char', 'amount': 29.8}, 'CF_IPCC_2021__climate_change__global_warming_potential_GWP100__Nitrogen_oxides__air__non_urban_air_or_from_high_stac

,scenario,msp_usd_per_kg,contribution_sum,closure_error,irr,passes_closure,passes_irr
0,s_c1,4.322083,4.322083,-4.378720e-12,0.1,True,True
1,s_c2,4.399352,4.399352,0.000000e+00,0.1,True,True
2,s_o1,6.005590,6.005590,8.881784e-16,0.1,True,True
3,s_o2,5.268109,5.268109,0.000000e+00,0.1,True,True


## 3. Export exact TEA main-cost-category results

These are the eight signed present-value-per-kg components shown in Figure 3. The sum of the categories is exactly the solved MSP. Negative byproduct credits remain negative.

In [4]:
breakdown_rows = []
for sc in SCENARIOS:
    item = tea_cache[sc]
    msp = float(item["result"].summary["msp_usd_per_kg"])
    for category, value in item["contributions"].items():
        value = float(value)
        breakdown_rows.append({
            "scenario": sc,
            "pathway": PATHWAY[sc],
            "category": category,
            "contribution_usd_per_kg": value,
            "contribution_pct_of_msp": 100.0 * value / msp,
            "msp_usd_per_kg": msp,
        })

tea_msp_breakdown = pd.DataFrame(breakdown_rows)
tea_msp_breakdown.to_csv(OUTPUT_DIR / "tea_msp_breakdown.csv", index=False)
display(tea_msp_breakdown.pivot(index="category", columns="pathway", values="contribution_usd_per_kg"))

pathway,BC1,BC2,BO1,BO2
category,,,,
Byproduct credits,-0.000000,-0.411463,-0.000000,-1.313932
CAPEX,0.229627,0.456938,1.037296,1.472114
Electricity,0.482971,0.525064,0.431549,0.426526
Fe catalyst,1.757577,1.757577,1.772196,1.772196
Feedstock,0.341083,0.341083,0.606368,0.606368
Finance & tax,0.009915,0.017408,0.037510,0.052880
Fixed OPEX,0.160269,0.236531,0.495973,0.641853
Other variable OPEX,1.340641,1.476213,1.624698,1.610105


## 4. Export physical TEA streams in native units

`flow_per_hr` is the quantity used by the TEA. `quantity_per_kg_graphite` normalizes each stream to the pathway's graphite production rate, so readers can interpret the physical basis without knowing plant scale. Native units are preserved (e.g., kWh, m³, kg).

In [5]:
stream_frames = []
for sc in SCENARIOS:
    item = tea_cache[sc]
    df = item["streams"].copy()
    flow_col = "flow_per_hr" if "flow_per_hr" in df.columns else "mass_flow_kg_hr"
    prod_mask = df["is_product"].astype(bool) & df[flow_col].gt(0)
    product_rate = float(df.loc[prod_mask, flow_col].sum())
    if product_rate <= 0:
        raise AssertionError(f"{sc}: no positive graphite product rate")
    # Standardize metadata columns; source tables may already contain them.
    df = df.drop(columns=["scenario", "pathway"], errors="ignore")
    df.insert(0, "pathway", PATHWAY[sc])
    df.insert(0, "scenario", sc)
    df["product_rate_kg_per_hr"] = product_rate
    df["quantity_per_kg_graphite"] = pd.to_numeric(df[flow_col], errors="raise") / product_rate
    df["quantity_per_kg_graphite_unit"] = df["flow_unit"].astype(str) + "/kg graphite"
    stream_frames.append(df)

tea_streams = pd.concat(stream_frames, ignore_index=True)
unit_counts = tea_streams.groupby(["scenario", "material_code"])["flow_unit"].nunique()
assert int((unit_counts > 1).sum()) == 0
for code_, expected in {"water":"m3", "NG":"m3", "electricity":"kWh"}.items():
    units = tea_streams.loc[tea_streams.material_code.eq(code_), "flow_unit"].dropna().unique().tolist()
    if units:
        assert units == [expected], (code_, units)

tea_streams.to_csv(OUTPUT_DIR / "tea_streams.csv", index=False)
print("TEA native-unit gate: PASS")
display(tea_streams.head(20))

TEA native-unit gate: PASS


,scenario,pathway,material_code,flow_per_hr,flow_unit,mass_flow_kg_hr,is_feedstock,is_product,is_byproduct,product_rate_kg_per_hr,quantity_per_kg_graphite,quantity_per_kg_graphite_unit
0,s_c1,BC1,Fe,43736.062081,kg,43736.062081,False,False,False,16814.747688,2.601054,kg/kg graphite
1,s_c1,BC1,HCl,15120.000748,kg,15120.000748,False,False,False,16814.747688,0.899211,kg/kg graphite
2,s_c1,BC1,N2,416.666687,kg,416.666687,False,False,False,16814.747688,0.024780,kg/kg graphite
3,s_c1,BC1,NaOH,3960.000196,kg,3960.000196,False,False,False,16814.747688,0.235508,kg/kg graphite
4,s_c1,BC1,biochar,21868.031040,kg,21868.031040,False,False,False,16814.747688,1.300527,kg/kg graphite
5,s_c1,BC1,diesel,408.650510,kg,408.650510,False,False,False,16814.747688,0.024303,kg/kg graphite
6,s_c1,BC1,electricity,93640.335353,kWh,93640.335353,False,False,False,16814.747688,5.568941,kWh/kg graphite
7,s_c1,BC1,feedstock,83333.337455,kg,83333.337455,True,False,False,16814.747688,4.955967,kg/kg graphite
8,s_c1,BC1,graphite,16814.747688,kg,16814.747688,False,True,False,16814.747688,1.000000,kg/kg graphite
9,s_c1,BC1,natural gas,2685.347885,m3,2685.347885,False,False,False,16814.747688,0.159702,m3/kg graphite


## 5. Export prices, CAPEX inputs, labor inputs, calculated capital basis, assumptions, and solved cash flows

The raw input tables are exported without manually retyping values. The capital summary and cash-flow tables provide an additional audit trail from equipment/capital assumptions through project financing to MSP.

In [6]:
def _tag(df, sc):
    """Return a copy with authoritative scenario/pathway metadata first.

    Some source DataFrames (notably CAPEX inputs) already carry a scenario
    column. Drop any pre-existing metadata columns before re-adding them so
    this helper is idempotent and safe on notebook reruns.
    """
    out = df.copy().drop(columns=["scenario", "pathway"], errors="ignore")
    out.insert(0, "pathway", PATHWAY[sc])
    out.insert(0, "scenario", sc)
    return out

tea_prices = pd.concat([_tag(tea_cache[sc]["bundle"].prices_df, sc) for sc in SCENARIOS], ignore_index=True)
tea_capex_inputs = pd.concat([_tag(tea_cache[sc]["bundle"].capex_df, sc) for sc in SCENARIOS], ignore_index=True)
tea_labour_inputs = pd.concat([_tag(tea_cache[sc]["bundle"].labour_df, sc) for sc in SCENARIOS], ignore_index=True)

tea_prices.to_csv(OUTPUT_DIR / "tea_prices.csv", index=False)
tea_capex_inputs.to_csv(OUTPUT_DIR / "tea_capex_inputs.csv", index=False)
tea_labour_inputs.to_csv(OUTPUT_DIR / "tea_labour_inputs.csv", index=False)

capital_rows = []
cashflow_frames = []
for sc in SCENARIOS:
    res = tea_cache[sc]["result"]
    cap = res.registry["capex_details"]
    capital_rows.append({
        "scenario": sc,
        "pathway": PATHWAY[sc],
        "isbl_installed_usd": float(cap.isbl),
        "osbl_installed_usd": float(cap.osbl),
        "total_installed_usd": float(cap.total_installed),
        "tdc_usd": float(cap.tdc),
        "total_indirect_capital_usd": float(cap.tic),
        "fci_usd": float(cap.fci),
        "working_capital_usd": float(cap.wc),
        "land_usd": float(cap.land),
        "total_project_capital_usd": float(cap.total_project_capital),
        "annual_maintenance_usd": float(res.summary["annual_maintenance_cost"]),
        "annual_insurance_property_tax_usd": float(res.summary["annual_insurance_etc_cost"]),
        "msp_usd_per_kg": float(res.summary["msp_usd_per_kg"]),
        "target_irr": float(res.summary["target_irr"]),
    })
    cf = res.cashflows.copy().reset_index()
    first = cf.columns[0]
    if first != "year":
        cf = cf.rename(columns={first:"year"})
    cf = _tag(cf, sc)
    cashflow_frames.append(cf)

tea_capital_summary = pd.DataFrame(capital_rows)
tea_cashflows = pd.concat(cashflow_frames, ignore_index=True)
tea_capital_summary.to_csv(OUTPUT_DIR / "tea_capital_summary.csv", index=False)
tea_cashflows.to_csv(OUTPUT_DIR / "tea_cashflows.csv", index=False)

display(tea_capital_summary)

,scenario,pathway,isbl_installed_usd,osbl_installed_usd,total_installed_usd,tdc_usd,total_indirect_capital_usd,fci_usd,working_capital_usd,land_usd,total_project_capital_usd,annual_maintenance_usd,annual_insurance_property_tax_usd,msp_usd_per_kg,target_irr
0,s_c1,BC1,1.288599e+08,7.377139e+06,1.362371e+08,1.587876e+08,9.527254e+07,2.540601e+08,1.270301e+07,1848000.0,2.686111e+08,7.621803e+06,2.540601e+06,4.322083,0.1
1,s_c2,BC2,2.348284e+08,4.122355e+07,2.760519e+08,3.171469e+08,1.902881e+08,5.074350e+08,2.537175e+07,1848000.0,5.346547e+08,1.522305e+07,5.074350e+06,4.399352,0.1
2,s_o1,BO1,2.761156e+08,8.086860e+07,3.569842e+08,4.053045e+08,2.431827e+08,6.484872e+08,3.242436e+07,1848000.0,6.827595e+08,1.945461e+07,6.484872e+06,6.005590,0.1
3,s_o2,BO2,3.790126e+08,1.303586e+08,5.093712e+08,5.756984e+08,3.454190e+08,9.211174e+08,4.605587e+07,1848000.0,9.690213e+08,2.763352e+07,9.211174e+06,5.268109,0.1


In [7]:
from graphite_sus.tea_models.assumptions import get_tea_assumptions

assumptions = get_tea_assumptions(TEA_PROFILE)
raw_assumptions = dataclasses.asdict(assumptions)

UNIT_HINTS = {
    "target_irr":"fraction", "equity_pct":"fraction", "loan_interest":"fraction",
    "loan_term_yr":"year", "tax_rate":"fraction", "project_years":"year",
    "operating_hours":"h/year", "warehouse_pct":"fraction", "site_dev_pct":"fraction",
    "add_pipe_pct":"fraction", "proratable_pct":"fraction", "field_exp_pct":"fraction",
    "home_office_pct":"fraction", "contingency_pct":"fraction", "other_pct":"fraction",
    "working_capital_pct":"fraction", "land_needed_acre":"acre", "cost_per_acre":"USD/acre",
    "maintenance_pct":"fraction", "insurance_property_tax_pct_fci":"fraction",
    "start_up_time_yr":"year", "capacity_ramp":"fraction", "variable_ramp":"fraction",
    "fixed_ramp":"fraction",
}

assumption_rows = []
for key, value in raw_assumptions.items():
    if key == "construction_schedule":
        for year, fraction in value:
            assumption_rows.append({
                "profile": TEA_PROFILE,
                "parameter": f"construction_fraction_year_{int(year):+d}",
                "value": float(fraction),
                "unit": "fraction",
            })
    else:
        assumption_rows.append({
            "profile": TEA_PROFILE,
            "parameter": key,
            "value": value,
            "unit": UNIT_HINTS.get(key, "text" if isinstance(value, str) else "fraction_or_model_setting"),
        })

tea_financial_assumptions = pd.DataFrame(assumption_rows)
tea_financial_assumptions.to_csv(OUTPUT_DIR / "tea_financial_assumptions.csv", index=False)
display(tea_financial_assumptions)

,profile,parameter,value,unit
0,literature_hybrid,name,literature_hybrid,text
1,literature_hybrid,target_irr,0.1,fraction
2,literature_hybrid,equity_pct,0.4,fraction
3,literature_hybrid,loan_interest,0.08,fraction
4,literature_hybrid,loan_term_yr,10,year
5,literature_hybrid,tax_rate,0.21,fraction
6,literature_hybrid,project_years,30,year
7,literature_hybrid,operating_hours,8000.0,h/year
8,literature_hybrid,construction_fraction_year_-2,0.08,fraction
9,literature_hybrid,construction_fraction_year_-1,0.6,fraction


## 6. TEA round-trip verification from the exported CSV files

Two independent table-level checks are performed:

1. Reload `tea_msp_breakdown.csv` and verify the eight signed categories sum to the authoritative MSP.
2. Reload the exported streams, prices, CAPEX, and labor CSVs and rerun `VectorModel` using the named financial-assumption profile. This proves the exported input tables are sufficient to reproduce the model result rather than merely reporting it.

A third audit recomputes NPV from the **reloaded solved cash-flow table** at the target IRR; NPV should be approximately zero.

In [8]:
from graphite_sus.tea_models.vector_model import VectorModel

# 6a. Breakdown-only closure after disk reload
_break = pd.read_csv(OUTPUT_DIR / "tea_msp_breakdown.csv")
_break_check = (
    _break.groupby(["scenario", "pathway", "msp_usd_per_kg"], as_index=False)["contribution_usd_per_kg"]
    .sum()
    .rename(columns={"contribution_usd_per_kg":"breakdown_sum_usd_per_kg"})
)
_break_check["breakdown_delta_usd_per_kg"] = _break_check["breakdown_sum_usd_per_kg"] - _break_check["msp_usd_per_kg"]
_break_check["breakdown_pass"] = _break_check["breakdown_delta_usd_per_kg"].abs() <= 1e-8
assert bool(_break_check["breakdown_pass"].all())

# 6b. Re-run TEA from reloaded exported inputs
_s = pd.read_csv(OUTPUT_DIR / "tea_streams.csv")
# Robustly restore Boolean flags after CSV round-trip.
for _bool_col in ["is_feedstock", "is_product", "is_byproduct"]:
    if _bool_col in _s.columns and _s[_bool_col].dtype == object:
        _s[_bool_col] = _s[_bool_col].astype(str).str.strip().str.lower().map({"true": True, "false": False})
        if _s[_bool_col].isna().any():
            raise ValueError(f"Could not restore Boolean TEA stream flag {_bool_col!r} after CSV reload")
_p = pd.read_csv(OUTPUT_DIR / "tea_prices.csv")
_c = pd.read_csv(OUTPUT_DIR / "tea_capex_inputs.csv")
_l = pd.read_csv(OUTPUT_DIR / "tea_labour_inputs.csv")

model = VectorModel(assumptions=get_tea_assumptions(TEA_PROFILE))
rerun_rows = []
for sc in SCENARIOS:
    def subset(df):
        x = df.loc[df["scenario"].eq(sc)].copy()
        return x.drop(columns=[c for c in ["scenario", "pathway"] if c in x.columns])

    streams_rt = subset(_s).drop(columns=[
        c for c in ["product_rate_kg_per_hr", "quantity_per_kg_graphite", "quantity_per_kg_graphite_unit"]
        if c in _s.columns
    ])
    prices_rt = subset(_p)
    capex_rt = subset(_c)
    labour_rt = subset(_l)

    res_rt = model.run_breakdown(
        streams_df=streams_rt,
        prices_df=prices_rt,
        capex_df=capex_rt,
        labour_df=labour_rt,
    )
    ref = tea_cache[sc]["result"]
    msp_ref = float(ref.summary["msp_usd_per_kg"])
    msp_rt = float(res_rt.summary["msp_usd_per_kg"])
    irr_rt = float(res_rt.summary["irr_at_debug_price"])
    rerun_rows.append({
        "scenario": sc,
        "pathway": PATHWAY[sc],
        "authoritative_msp_usd_per_kg": msp_ref,
        "reloaded_input_msp_usd_per_kg": msp_rt,
        "msp_delta_usd_per_kg": msp_rt - msp_ref,
        "reloaded_input_irr": irr_rt,
        "msp_roundtrip_pass": bool(np.isclose(msp_rt, msp_ref, rtol=0, atol=1e-8)),
        "irr_roundtrip_pass": bool(np.isclose(irr_rt, float(assumptions.target_irr), rtol=0, atol=1e-6)),
    })

tea_input_roundtrip = pd.DataFrame(rerun_rows)
assert bool(tea_input_roundtrip["msp_roundtrip_pass"].all())
assert bool(tea_input_roundtrip["irr_roundtrip_pass"].all())

# 6c. Independent NPV audit from reloaded solved cash flows
_cf = pd.read_csv(OUTPUT_DIR / "tea_cashflows.csv")
npv_rows = []
for sc in SCENARIOS:
    x = _cf.loc[_cf["scenario"].eq(sc)].copy()
    years = pd.to_numeric(x["year"], errors="raise").to_numpy(float)
    ncf = pd.to_numeric(x["net_cash_flow"], errors="raise").to_numpy(float)
    r = float(tea_cache[sc]["result"].summary["target_irr"])
    npv = float(np.sum(ncf / (1.0 + r) ** years))
    fci = float(tea_cache[sc]["result"].registry["capex_details"].fci)
    tol = max(0.05, 1e-9 * abs(fci))
    npv_rows.append({
        "scenario": sc, "pathway": PATHWAY[sc], "npv_at_target_irr_usd": npv,
        "npv_abs_tolerance_usd": tol, "cashflow_npv_pass": abs(npv) <= tol,
    })
npv_audit = pd.DataFrame(npv_rows)
assert bool(npv_audit["cashflow_npv_pass"].all())

tea_validation = _break_check.merge(tea_input_roundtrip, on=["scenario","pathway"], how="outer").merge(npv_audit, on=["scenario","pathway"], how="outer")
tea_validation.to_csv(OUTPUT_DIR / "tea_validation.csv", index=False)
display(tea_validation)
print("TEA exported-table round trip: PASS")

,scenario,pathway,msp_usd_per_kg,breakdown_sum_usd_per_kg,breakdown_delta_usd_per_kg,breakdown_pass,authoritative_msp_usd_per_kg,reloaded_input_msp_usd_per_kg,msp_delta_usd_per_kg,reloaded_input_irr,msp_roundtrip_pass,irr_roundtrip_pass,npv_at_target_irr_usd,npv_abs_tolerance_usd,cashflow_npv_pass
0,s_c1,BC1,4.322083,4.322083,-4.378720e-12,True,4.322083,4.322083,0.000000e+00,0.1,True,True,5.485136e-03,0.254060,True
1,s_c2,BC2,4.399352,4.399352,0.000000e+00,True,4.399352,4.399352,0.000000e+00,0.1,True,True,-2.114102e-07,0.507435,True
2,s_o1,BO1,6.005590,6.005590,8.881784e-16,True,6.005590,6.005590,-8.881784e-16,0.1,True,True,1.210719e-08,0.648487,True
3,s_o2,BO2,5.268109,5.268109,0.000000e+00,True,5.268109,5.268109,-1.776357e-15,0.1,True,True,2.905726e-07,0.921117,True


TEA exported-table round trip: PASS


## 7. Export factor-agnostic foreground LCI

This table is built from the **collapsed LCA system activity**, not the TEA stream table. Each non-production exchange is evaluated using the current activity-parameter environment. Technosphere rows represent required background services/materials; biosphere rows represent elementary flows.

Internal `graphite_system/link::*` providers are retained for traceability but marked `include_in_lcia=False`, because their environmental burden is already represented elsewhere in the collapsed system. Assigning a new external factor to those rows would double count.

In [9]:
import bw2data as bd
from bw2data.parameters import ActivityParameter
from graphite_sus.lca_breakdown_factors import current_factor, method_slug, parameter_amount
from graphite_sus.plots.lca_multiimpact import DEFAULT_MULTIIMPACT_METHODS

bd.parameters.recalculate()
METHODS = list(DEFAULT_MULTIIMPACT_METHODS)

def _eval_exchange_amount(exc, env):
    expr = exc.get("formula")
    if expr:
        try:
            return float(eval(expr, {"__builtins__": {}}, env))
        except Exception as exc_eval:
            raise RuntimeError(f"Could not evaluate exchange formula {expr!r}") from exc_eval
    return float(exc.get("amount") or 0.0)

def _stage_from_comment(exc):
    comment = str(exc.get("comment") or "")
    m = re.search(r"stage=([^;]+)", comment)
    return m.group(1).strip() if m else "foreground"

def _node_key(node):
    key = getattr(node, "key", None)
    if key is not None:
        return tuple(key)
    return (node.get("database"), node.get("code"))

def _portable_key(kind, node):
    name = str(node.get("name") or node.get("code") or "")
    unit = str(node.get("unit") or "")
    if kind == "technosphere":
        context = str(node.get("location") or "GLO")
    else:
        cats = node.get("categories") or ()
        context = " > ".join(map(str, cats)) if isinstance(cats, (list, tuple)) else str(cats)
    return f"{kind}::{name}::{context}::{unit}"

lci_rows = []
factor_rows = []
allocation_rows = []
expected_rows = []

for sc in SCENARIOS:
    act = tea_cache[sc]["bundle"].lca_sys_act
    sys_db, sys_code = act.key
    ap = ActivityParameter.load(f"{sys_db}::{sys_code}")
    env = {str(k): float(v.get("amount", 0.0)) for k, v in ap.items()}

    h2_kg = parameter_amount(ap, "byprod__h2__kg") if "byprod__h2__kg" in ap else 0.0
    hc_kg = parameter_amount(ap, "byprod__hydrocarbon__kg") if "byprod__hydrocarbon__kg" in ap else 0.0
    mass_fraction = 1.0 / (1.0 + h2_kg + hc_kg)

    method_fraction_checks = []
    for method in METHODS:
        slug = method_slug(method)
        system_impact = parameter_amount(ap, f"impact__{slug}")
        allocated_impact = parameter_amount(ap, f"impact_alloc__mass__{slug}")
        ratio = allocated_impact / system_impact if abs(system_impact) > 1e-15 else np.nan
        method_fraction_checks.append(ratio)
        expected_rows.append({
            "scenario": sc,
            "pathway": PATHWAY[sc],
            "method_slug": slug,
            "impact_category": method[1],
            "authoritative_system_impact": system_impact,
            "authoritative_mass_allocated_impact": allocated_impact,
            "mass_allocation_fraction": mass_fraction,
        })

    finite_ratios = [x for x in method_fraction_checks if np.isfinite(x)]
    if finite_ratios:
        assert np.allclose(finite_ratios, mass_fraction, rtol=1e-9, atol=1e-10)

    allocation_rows.append({
        "scenario": sc,
        "pathway": PATHWAY[sc],
        "h2_coproduct_kg_per_kg_graphite_system": h2_kg,
        "hydrocarbon_coproduct_kg_per_kg_graphite_system": hc_kg,
        "mass_allocation_fraction_to_graphite": mass_fraction,
        "ratio_check_min": min(finite_ratios) if finite_ratios else np.nan,
        "ratio_check_max": max(finite_ratios) if finite_ratios else np.nan,
    })

    for j, exc in enumerate(act.exchanges()):
        etype = str(exc.get("type") or "")
        if etype == "production":
            continue
        if etype not in {"technosphere", "biosphere"}:
            warnings.warn(f"{sc}: skipping unsupported exchange type {etype!r}")
            continue

        node = bd.get_node(key=exc.input.key if hasattr(exc.input, "key") else tuple(exc.input))
        db, code_ = _node_key(node)
        qty = _eval_exchange_amount(exc, env)
        stage = _stage_from_comment(exc)
        name = str(node.get("name") or node.get("code") or "")
        unit = str(node.get("unit") or "")
        location = str(node.get("location") or "") if etype == "technosphere" else ""
        cats = node.get("categories") or ()
        categories = " > ".join(map(str, cats)) if isinstance(cats, (list, tuple)) else str(cats or "")
        flow_key = f"{etype}|{db}|{code_}"
        portable_key = _portable_key(etype, node)
        internal_zero_link = bool(etype == "technosphere" and db == sys_db and str(code_).startswith("link::"))
        include = not internal_zero_link

        lci_rows.append({
            "scenario": sc,
            "pathway": PATHWAY[sc],
            "functional_unit": f"1 {act.get('unit') or 'unit'} {act.get('reference product') or act.get('name')}",
            "system_activity_database": sys_db,
            "system_activity_code": sys_code,
            "exchange_index": j,
            "exchange_type": etype,
            "stage": stage,
            "material_code": exc.get("material_code") or "",
            "flow_key": flow_key,
            "portable_mapping_key": portable_key,
            "provider_or_flow_name": name,
            "provider_or_flow_database": db,
            "provider_or_flow_code": code_,
            "provider_location": location,
            "biosphere_categories": categories,
            "quantity_per_functional_unit": qty,
            "flow_unit": unit,
            "mass_allocation_fraction": mass_fraction,
            "mass_allocated_quantity_per_functional_unit": qty * mass_fraction,
            "exchange_formula": exc.get("formula") or "",
            "exchange_comment": exc.get("comment") or "",
            "is_internal_zero_impact_link": internal_zero_link,
            "include_in_lcia": include,
        })

        for method in METHODS:
            slug = method_slug(method)
            impact_unit = str((bd.Method(method).metadata or {}).get("unit") or "")
            if internal_zero_link:
                factor = 0.0
                factor_parameter = "internal_zero_link"
            else:
                factor, factor_parameter = current_factor(ap, method, node, etype)
            factor_rows.append({
                "scenario": sc,
                "pathway": PATHWAY[sc],
                "flow_key": flow_key,
                "portable_mapping_key": portable_key,
                "exchange_type": etype,
                "provider_or_flow_name": name,
                "provider_or_flow_database": db,
                "provider_or_flow_code": code_,
                "provider_location": location,
                "biosphere_categories": categories,
                "flow_unit": unit,
                "method_slug": slug,
                "method": " | ".join(map(str, method)),
                "impact_category": method[1],
                "impact_unit": impact_unit,
                "reference_unit_impact_factor": float(factor),
                "reference_factor_parameter": factor_parameter,
                "factor_unit": f"{impact_unit}/{unit}" if impact_unit and unit else "",
                "include_in_lcia": include,
            })

lci_foreground = pd.DataFrame(lci_rows)
lci_allocation_audit = pd.DataFrame(allocation_rows)
reference_factors = pd.DataFrame(factor_rows).drop_duplicates(
    subset=["scenario", "flow_key", "method_slug"], keep="first"
).reset_index(drop=True)
expected_impacts = pd.DataFrame(expected_rows)

# Verify duplicated factor definitions, if any, were actually consistent before de-duplication.
_factor_all = pd.DataFrame(factor_rows)
_consistency = _factor_all.groupby(["scenario","flow_key","method_slug"])["reference_unit_impact_factor"].nunique(dropna=False)
assert int((_consistency > 1).sum()) == 0

lci_foreground.to_csv(OUTPUT_DIR / "lci_foreground_inventory.csv", index=False)
lci_allocation_audit.to_csv(OUTPUT_DIR / "lci_allocation_audit.csv", index=False)
reference_factors.to_csv(OUTPUT_DIR / "lcia_factor_mapping_reference.csv", index=False)

template = reference_factors.drop(columns=["reference_unit_impact_factor", "reference_factor_parameter"]).copy()
template["user_factor"] = np.nan
template["user_factor_source"] = ""
template["user_factor_notes"] = ""
template.to_csv(OUTPUT_DIR / "lcia_factor_mapping_template.csv", index=False)

display(lci_allocation_audit)
display(lci_foreground.head(20))

,scenario,pathway,h2_coproduct_kg_per_kg_graphite_system,hydrocarbon_coproduct_kg_per_kg_graphite_system,mass_allocation_fraction_to_graphite,ratio_check_min,ratio_check_max
0,s_c1,BC1,0.000000,0.000000,1.000000,1.000000,1.000000
1,s_c2,BC2,0.017956,0.338846,0.737027,0.737027,0.737027
2,s_o1,BO1,0.002203,0.000000,0.997802,0.997802,0.997802
3,s_o2,BO2,0.090308,0.718062,0.552984,0.552984,0.552984


,scenario,pathway,functional_unit,system_activity_database,system_activity_code,exchange_index,exchange_type,stage,material_code,flow_key,...,provider_location,biosphere_categories,quantity_per_functional_unit,flow_unit,mass_allocation_fraction,mass_allocated_quantity_per_functional_unit,exchange_formula,exchange_comment,is_internal_zero_impact_link,include_in_lcia
0,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,1,technosphere,"slow pyrolysis, biochar production, no byproduct",diesel,technosphere|ei39_cutoff|b613a0e604fce45d41460...,...,RoW,,0.007137,kilogram,1.0,0.007137,tech__slow_pyrolysis_biochar_production_no_byp...,"stage=slow pyrolysis, biochar production, no b...",False,True
1,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,2,technosphere,"slow pyrolysis, biochar production, no byproduct",N2,technosphere|ei39_cutoff|2ba127601085d3c181813...,...,RoW,,0.024780,kilogram,1.0,0.024780,(0.024779835832075225)*stage_scale__slow_pyrol...,"stage=slow pyrolysis, biochar production, no b...",False,True
2,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,3,technosphere,"slow pyrolysis, biochar production, no byproduct",electricity,technosphere|ei39_cutoff|2df74714a5a7c8cc88d70...,...,US,,0.594027,kilowatt hour,1.0,0.594027,tech__slow_pyrolysis_biochar_production_no_byp...,"stage=slow pyrolysis, biochar production, no b...",False,True
3,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,4,technosphere,"treatment of waste water, waste water, to emit...",,technosphere|ei39_cutoff|c2656868cbf6ba64e0236...,...,RoW,,-0.000898,cubic meter,1.0,-0.000898,(-0.0008984972462837766)*stage_scale__treatmen...,"stage=treatment of waste water, waste water, t...",False,True
4,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,5,technosphere,"residues production, forestry residue",,technosphere|ei39_cutoff|289eaaf5e6f8ef0f5df16...,...,RoW,,0.264318,kilogram,1.0,0.264318,(0.2643182365144131)*stage_scale__residues_pro...,"stage=residues production, forestry residue; p...",False,True
5,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,6,technosphere,"residues production, forestry residue",,technosphere|ei39_cutoff|cc629a57d77e951cdca1e...,...,RoW,,0.264318,kilogram,1.0,0.264318,(0.2643182365144131)*stage_scale__residues_pro...,"stage=residues production, forestry residue; p...",False,True
6,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,7,technosphere,"residues production, forestry residue",,technosphere|ei39_cutoff|39637a65852c08973cb0b...,...,RoW,,0.264318,kilogram,1.0,0.264318,(0.2643182365144131)*stage_scale__residues_pro...,"stage=residues production, forestry residue; p...",False,True
7,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,8,technosphere,"residues production, forestry residue",,technosphere|ei39_cutoff|c655f5b0ff84702a0077a...,...,RoW,,0.264318,kilogram,1.0,0.264318,(0.2643182365144131)*stage_scale__residues_pro...,"stage=residues production, forestry residue; p...",False,True
8,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,9,technosphere,"residues production, forestry residue",,technosphere|ei39_cutoff|7f63fa1d975fd20397933...,...,RoW,,0.264318,kilogram,1.0,0.264318,(0.2643182365144131)*stage_scale__residues_pro...,"stage=residues production, forestry residue; p...",False,True
9,s_c1,BC1,1 kilogram graphite,graphite_system,system::graphite_prod_char::graphtz_char,10,technosphere,"residues production, forestry residue",,technosphere|ei39_cutoff|a54115ba044e5f4394c78...,...,RoW,,0.264318,kilogram,1.0,0.264318,(0.2643182365144131)*stage_scale__residues_pro...,"stage=residues production, forestry residue; p...",False,True


## 8. LCIA round trip: exported LCI × exported reference factors

This check **reloads both CSV files from disk**. For each pathway and each of the six paper impact methods, it multiplies the physical exchange quantity by the corresponding current unit-impact factor / CF, sums all included exchanges, then applies the independently exported graphite mass-allocation fraction.

Passing all 24 cases demonstrates that the public LCI table contains the physical information needed to reproduce the paper's deterministic LCIA when equivalent factors are mapped.

In [10]:
def calculate_lcia_from_mapping(lci_df, factor_df, *, factor_col, allocation="mass"):
    keys = ["scenario", "pathway", "flow_key"]
    merged = lci_df.merge(
        factor_df,
        on=keys,
        how="left",
        suffixes=("_lci", "_factor"),
        validate="many_to_many",
    )
    include_col = "include_in_lcia_lci" if "include_in_lcia_lci" in merged else "include_in_lcia"
    merged = merged.loc[merged[include_col].astype(bool)].copy()
    if merged[factor_col].isna().any():
        missing = merged.loc[merged[factor_col].isna(), ["pathway","flow_key","provider_or_flow_name_lci"]].drop_duplicates()
        raise ValueError(f"Missing mapped factors for included LCI rows:\n{missing.to_string(index=False)}")

    merged["unallocated_contribution"] = (
        pd.to_numeric(merged["quantity_per_functional_unit"], errors="raise")
        * pd.to_numeric(merged[factor_col], errors="raise")
    )
    if allocation == "mass":
        merged["allocated_contribution"] = (
            merged["unallocated_contribution"]
            * pd.to_numeric(merged["mass_allocation_fraction"], errors="raise")
        )
    elif allocation in {None, "none", "system"}:
        merged["allocated_contribution"] = merged["unallocated_contribution"]
    else:
        raise ValueError("allocation must be 'mass' or 'system'")

    out = (
        merged.groupby(["scenario","pathway","method_slug","impact_category","impact_unit"], as_index=False)
        .agg(
            reconstructed_system_impact=("unallocated_contribution","sum"),
            reconstructed_mass_allocated_impact=("allocated_contribution","sum"),
        )
    )
    return out, merged

_lci_disk = pd.read_csv(OUTPUT_DIR / "lci_foreground_inventory.csv")
_fac_disk = pd.read_csv(OUTPUT_DIR / "lcia_factor_mapping_reference.csv")
reconstructed, lci_factor_detail = calculate_lcia_from_mapping(
    _lci_disk, _fac_disk,
    factor_col="reference_unit_impact_factor",
    allocation="mass",
)

lci_validation = reconstructed.merge(
    expected_impacts,
    on=["scenario","pathway","method_slug","impact_category"],
    how="left",
    validate="one_to_one",
)
lci_validation["system_delta"] = lci_validation["reconstructed_system_impact"] - lci_validation["authoritative_system_impact"]
lci_validation["mass_allocated_delta"] = lci_validation["reconstructed_mass_allocated_impact"] - lci_validation["authoritative_mass_allocated_impact"]
lci_validation["system_closure_pass"] = np.isclose(
    lci_validation["reconstructed_system_impact"],
    lci_validation["authoritative_system_impact"],
    rtol=1e-9, atol=1e-10,
)
lci_validation["mass_allocated_closure_pass"] = np.isclose(
    lci_validation["reconstructed_mass_allocated_impact"],
    lci_validation["authoritative_mass_allocated_impact"],
    rtol=1e-9, atol=1e-10,
)

assert len(lci_validation) == len(SCENARIOS) * len(METHODS) == 24
assert bool(lci_validation["system_closure_pass"].all())
assert bool(lci_validation["mass_allocated_closure_pass"].all())
lci_validation.to_csv(OUTPUT_DIR / "lci_validation.csv", index=False)
print("LCI × reference-factor round trip: PASS (24/24)")
display(lci_validation)

LCI × reference-factor round trip: PASS (24/24)


,scenario,pathway,method_slug,impact_category,impact_unit,reconstructed_system_impact,reconstructed_mass_allocated_impact,authoritative_system_impact,authoritative_mass_allocated_impact,mass_allocation_fraction,system_delta,mass_allocated_delta,system_closure_pass,mass_allocated_closure_pass
0,s_c1,BC1,IPCC_2021__climate_change__global_warming_pote...,climate change,kg CO2-Eq,5.191801,5.191801,5.191801,5.191801,1.000000,-1.776357e-15,-1.776357e-15,True,True
1,s_c1,BC1,TRACI_v2_1__acidification__acidification_poten...,acidification,kg SO2-Eq,0.019165,0.019165,0.019165,0.019165,1.000000,-7.112366e-16,-7.112366e-16,True,True
2,s_c1,BC1,TRACI_v2_1__ecotoxicity_freshwater__ecotoxicit...,ecotoxicity: freshwater,CTUe,95.939231,95.939231,95.939231,95.939231,1.000000,-1.421085e-14,-1.421085e-14,True,True
3,s_c1,BC1,TRACI_v2_1__eutrophication__eutrophication_pot...,eutrophication,kg N-Eq,0.024280,0.024280,0.024280,0.024280,1.000000,-8.465451e-16,-8.465451e-16,True,True
4,s_c1,BC1,TRACI_v2_1__particulate_matter_formation__part...,particulate matter formation,PM2.5-Eq,0.006867,0.006867,0.006867,0.006867,1.000000,-4.917941e-16,-4.917941e-16,True,True
5,s_c1,BC1,TRACI_v2_1__photochemical_oxidant_formation__m...,photochemical oxidant formation,kg O3-Eq,0.281970,0.281970,0.281970,0.281970,1.000000,-3.885781e-16,-3.885781e-16,True,True
6,s_c2,BC2,IPCC_2021__climate_change__global_warming_pote...,climate change,kg CO2-Eq,7.088036,5.224075,7.088036,5.224075,0.737027,-8.881784e-16,-8.881784e-16,True,True
7,s_c2,BC2,TRACI_v2_1__acidification__acidification_poten...,acidification,kg SO2-Eq,0.023059,0.016995,0.023059,0.016995,0.737027,-7.251144e-16,-5.308254e-16,True,True
8,s_c2,BC2,TRACI_v2_1__ecotoxicity_freshwater__ecotoxicit...,ecotoxicity: freshwater,CTUe,99.090552,73.032424,99.090552,73.032424,0.737027,-4.263256e-14,-2.842171e-14,True,True
9,s_c2,BC2,TRACI_v2_1__eutrophication__eutrophication_pot...,eutrophication,kg N-Eq,0.025539,0.018823,0.025539,0.018823,0.737027,-8.847090e-16,-6.557255e-16,True,True


## 9. Optional strong cross-check: reproduce spatial LCIA by replacing only unit factors

When the spatial source-map and regenerated electricity/diesel/natural-gas factor tables are available, this section performs a stronger test: it keeps **the exact same exported physical LCI**, replaces only the relevant unit-impact factors with state-specific values, and compares the result against the production `StateLCACalculator` formula evaluator.

This is the direct test of the claim that the LCI can be reused with a different factor mapping. It is optional only because `state_source_map` can live outside the repository.

In [11]:
from graphite_sus.spatial.config import SpatialPaths
from graphite_sus.spatial.emission_factors import StateSourceMap, EFTables
from graphite_sus.spatial.lca import StateLCACalculator

SPATIAL_PATHS = SpatialPaths()
_spatial_required = [
    SPATIAL_PATHS.state_source_map,
    SPATIAL_PATHS.electricity_ef,
    SPATIAL_PATHS.diesel_ef,
    SPATIAL_PATHS.natural_gas_ef,
]
spatial_crosscheck_ran = all(Path(p).exists() for p in _spatial_required)
spatial_crosscheck_pass = None
spatial_validation = pd.DataFrame()

if spatial_crosscheck_ran:
    state_map = StateSourceMap.from_csv(SPATIAL_PATHS.state_source_map)
    ef_tables = EFTables.from_csvs(
        SPATIAL_PATHS.electricity_ef,
        SPATIAL_PATHS.diesel_ef,
        SPATIAL_PATHS.natural_gas_ef,
    )
    calc = StateLCACalculator(state_map, ef_tables)
    states = sorted(map(str, state_map.by_state.index))
    spatial_rows = []

    for state in states:
        for sc in SCENARIOS:
            act = tea_cache[sc]["bundle"].lca_sys_act
            overrides = calc.build_overrides(act, state, METHODS)

            fac_state = reference_factors.loc[reference_factors["scenario"].eq(sc)].copy()
            fac_state["state_unit_impact_factor"] = fac_state["reference_unit_impact_factor"]
            mask = fac_state["reference_factor_parameter"].isin(overrides)
            fac_state.loc[mask, "state_unit_impact_factor"] = (
                fac_state.loc[mask, "reference_factor_parameter"].map(overrides).astype(float)
            )

            lci_sc = lci_foreground.loc[lci_foreground["scenario"].eq(sc)].copy()
            rec_state, _ = calculate_lcia_from_mapping(
                lci_sc,
                fac_state,
                factor_col="state_unit_impact_factor",
                allocation="mass",
            )
            expected_state = calc.evaluate(
                act, state, METHODS,
                allocation_scheme="mass",
                use_display=False,
            )

            for _, row in rec_state.iterrows():
                cat = row["impact_category"]
                reconstructed = float(row["reconstructed_mass_allocated_impact"])
                expected = float(expected_state[cat])
                spatial_rows.append({
                    "state": state,
                    "scenario": sc,
                    "pathway": PATHWAY[sc],
                    "impact_category": cat,
                    "reconstructed_from_exported_lci": reconstructed,
                    "authoritative_state_formula": expected,
                    "delta": reconstructed - expected,
                    "pass": bool(np.isclose(reconstructed, expected, rtol=1e-9, atol=1e-10)),
                    "n_factor_overrides": int(mask.sum()),
                })

    spatial_validation = pd.DataFrame(spatial_rows)
    spatial_crosscheck_pass = bool(spatial_validation["pass"].all())
    spatial_validation.to_csv(
        OUTPUT_DIR / "lci_spatial_factor_substitution_validation.csv", index=False
    )
    assert spatial_crosscheck_pass
    print(
        "Spatial LCI factor-substitution cross-check: PASS",
        f"({len(spatial_validation)} state×pathway×impact cases)"
    )
else:
    missing_spatial = [str(p) for p in _spatial_required if not Path(p).exists()]
    print("Spatial cross-check skipped because these inputs are unavailable:")
    for p in missing_spatial:
        print(" -", p)


Spatial LCI factor-substitution cross-check: PASS (1224 state×pathway×impact cases)


## 10. Example: use a user-supplied factor mapping

After filling `user_factor` in `lcia_factor_mapping_template.csv`, the same calculation helper can be used without changing the LCI. Factor units must be compatible with `flow_unit`. Technosphere factors should be **unit cradle-to-gate impact factors** for the mapped material/service; biosphere factors should be the appropriate LCIA characterization factors.

The cell below is intentionally non-failing while the template is blank.

In [12]:
user_map = pd.read_csv(OUTPUT_DIR / "lcia_factor_mapping_template.csv")
if user_map["user_factor"].notna().all():
    user_results, user_detail = calculate_lcia_from_mapping(
        pd.read_csv(OUTPUT_DIR / "lci_foreground_inventory.csv"),
        user_map,
        factor_col="user_factor",
        allocation="mass",
    )
    display(user_results)
else:
    n_missing = int(user_map["user_factor"].isna().sum())
    print(f"User mapping template is intentionally blank: {n_missing} factor rows still need user_factor values.")

User mapping template is intentionally blank: 978 factor rows still need user_factor values.


## 11. Table dictionary, hashes, and final acceptance gate

The manifest hashes both source workbooks and every reproducibility output, so a future manuscript/SI revision can identify exactly which files were used. The notebook fails if TEA or LCI round-trip closure is lost.

In [13]:
TABLES = {
    "tea_msp_breakdown.csv": "Exact signed Figure-3 cost-category contributions in USD/kg and percent of MSP.",
    "tea_streams.csv": "LCI-compiled physical TEA streams in native units plus normalized quantity per kg graphite.",
    "tea_prices.csv": "Deterministic pathway price table used by the TEA.",
    "tea_capex_inputs.csv": "Pathway-specific raw CAPEX input rows used by the TEA.",
    "tea_labour_inputs.csv": "Labor input rows used by the TEA.",
    "tea_financial_assumptions.csv": "Named literature_hybrid financing, tax, schedule, capital-adder, and OPEX assumptions.",
    "tea_capital_summary.csv": "Calculated ISBL/OSBL/TDC/indirect/FCI/WC/land capital basis and fixed-cost summaries.",
    "tea_cashflows.csv": "Solved annual project cash flows at pathway MSP.",
    "tea_validation.csv": "Disk-reload cost closure, input-table TEA rerun, IRR, and NPV checks.",
    "lci_foreground_inventory.csv": "Physical foreground/background-service requirements from the collapsed LCA system, factor agnostic.",
    "lci_allocation_audit.csv": "Coproduct masses and graphite mass-allocation fraction.",
    "lcia_factor_mapping_reference.csv": "Exact paper UF/CF values used only for deterministic round-trip verification.",
    "lcia_factor_mapping_template.csv": "Portable factor-map template with blank user_factor values.",
    "lci_validation.csv": "24 deterministic LCI×factor round-trip checks against authoritative system and mass-allocated impacts.",
    "fuel_ef_reference_audit.csv": "Option-B deterministic fuel-factor reference audit.",
    "electricity_ef_reference_audit.csv": "National electricity LCIA-method consistency audit.",
}
if (OUTPUT_DIR / "lci_spatial_factor_substitution_validation.csv").exists():
    TABLES["lci_spatial_factor_substitution_validation.csv"] = (
        "Optional strong check that exported physical LCI plus state-specific factor substitution reproduces the production spatial LCIA formula."
    )

table_dictionary = pd.DataFrame([{"file": k, "description": v} for k, v in TABLES.items()])
table_dictionary.to_csv(OUTPUT_DIR / "table_dictionary.csv", index=False)
TABLES["table_dictionary.csv"] = "Dictionary describing all reproducibility outputs."

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

required = list(TABLES)
missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
checks = {
    "authoritative_tea_closure": bool(tea_authoritative_validation["passes_closure"].all()),
    "authoritative_tea_irr": bool(tea_authoritative_validation["passes_irr"].all()),
    "exported_tea_breakdown_closure": bool(tea_validation["breakdown_pass"].all()),
    "exported_tea_input_msp_roundtrip": bool(tea_validation["msp_roundtrip_pass"].all()),
    "exported_tea_input_irr_roundtrip": bool(tea_validation["irr_roundtrip_pass"].all()),
    "exported_tea_cashflow_npv": bool(tea_validation["cashflow_npv_pass"].all()),
    "lci_system_impact_closure_24_of_24": bool(lci_validation["system_closure_pass"].all()),
    "lci_mass_allocated_closure_24_of_24": bool(lci_validation["mass_allocated_closure_pass"].all()),
    "allocation_fraction_method_invariant": bool(np.allclose(
        lci_allocation_audit["ratio_check_min"], lci_allocation_audit["ratio_check_max"],
        rtol=1e-9, atol=1e-10, equal_nan=True,
    )),
    "fuel_reference_gate": bool(fuel_ef_audit["passed"].all()),
    "electricity_reference_gate": bool(electricity_ef_audit["passed"].all()),
    "all_required_tables_written": not missing,
}
spatial_status = {
    "ran": bool(spatial_crosscheck_ran),
    "passed": None if spatial_crosscheck_pass is None else bool(spatial_crosscheck_pass),
    "n_cases": int(len(spatial_validation)),
}

manifest = {
    "project_name": PROJECT_NAME,
    "tea_profile": TEA_PROFILE,
    "electricity_mode": ELECTRICITY_MODE,
    "rebuild_databases": REBUILD_DATABASES,
    "source_files": {
        "lci_workbook": {"path": str(LCI_XLSX), "sha256": sha256(LCI_XLSX)},
        "tea_workbook": {"path": str(TEA_XLSX), "sha256": sha256(TEA_XLSX)},
    },
    "reference_manifests": {
        "fuel_ef_reference": fuel_reference_manifest(),
        "electricity_ef_reference": electricity_ef_reference_manifest(),
    },
    "outputs": {
        name: {"description": TABLES[name], "sha256": sha256(OUTPUT_DIR / name)}
        for name in required if (OUTPUT_DIR / name).exists()
    },
    "checks": checks,
    "optional_spatial_crosscheck": spatial_status,
    "missing_outputs": missing,
}
(OUTPUT_DIR / "reproducibility_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(checks.values()), manifest
print("REPRODUCIBILITY TABLE WORKFLOW ACCEPTED")

authoritative_tea_closure: PASS
authoritative_tea_irr: PASS
exported_tea_breakdown_closure: PASS
exported_tea_input_msp_roundtrip: PASS
exported_tea_input_irr_roundtrip: PASS
exported_tea_cashflow_npv: PASS
lci_system_impact_closure_24_of_24: PASS
lci_mass_allocated_closure_24_of_24: PASS
allocation_fraction_method_invariant: PASS
fuel_reference_gate: PASS
electricity_reference_gate: PASS
all_required_tables_written: PASS
REPRODUCIBILITY TABLE WORKFLOW ACCEPTED


In [14]:
# ============================================================
# Generate revised SI Table S17 from notebook 01b outputs
# ============================================================

import pandas as pd
import numpy as np

# OUTPUT_DIR is already defined earlier in notebook 01b as:
# PROJECT_ROOT / "result" / "reproducibility"

PATHWAY_ORDER = ["BC1", "BC2", "BO1", "BO2"]

# ------------------------------------------------------------
# 1. Load reproducibility outputs
# ------------------------------------------------------------
streams = pd.read_csv(OUTPUT_DIR / "tea_streams.csv")
capital = pd.read_csv(OUTPUT_DIR / "tea_capital_summary.csv")
cashflows = pd.read_csv(OUTPUT_DIR / "tea_cashflows.csv")
assumptions = pd.read_csv(OUTPUT_DIR / "tea_financial_assumptions.csv")


# ------------------------------------------------------------
# 2. Get annual operating hours from TEA assumptions
#    Do not hard-code 8000 so Table S17 stays synchronized
#    if the TEA assumption is changed later.
# ------------------------------------------------------------
operating_hours = float(
    assumptions.loc[
        assumptions["parameter"].eq("operating_hours"),
        "value"
    ].iloc[0]
)

print(f"Operating hours used: {operating_hours:,.0f} h/yr")


# ------------------------------------------------------------
# 3. Annual graphite production
#
# tea_streams.csv reports graphite production in kg/h.
#
# thousand metric tonne/yr =
# kg/h * h/yr / 1e6
# ------------------------------------------------------------
streams["material_code_norm"] = (
    streams["material_code"]
    .astype(str)
    .str.strip()
    .str.lower()
)

graphite_rate = (
    streams.loc[
        streams["is_product"].astype(bool),
        ["pathway", "flow_per_hr"]
    ]
    .groupby("pathway")["flow_per_hr"]
    .sum()
    .reindex(PATHWAY_ORDER)
)

annual_graphite_kt = graphite_rate * operating_hours / 1e6


# ------------------------------------------------------------
# 4. Biochar / biocoke precursor throughput
#
# BC pathways -> biochar
# BO pathways -> coke
# ------------------------------------------------------------
precursor = streams.loc[
    streams["material_code_norm"].isin(["biochar", "coke"]),
    ["pathway", "material_code_norm", "flow_per_hr"]
].copy()

precursor_rate = (
    precursor
    .groupby("pathway")["flow_per_hr"]
    .sum()
    .reindex(PATHWAY_ORDER)
)

# express as thousand kg/h for a compact SI table
precursor_rate_thousand = precursor_rate / 1e3


# ------------------------------------------------------------
# 5. Capital investment
#
# Use FCI rather than "average annual CAPEX".
# FCI = TDC + total indirect capital
# ------------------------------------------------------------
capital_by_path = (
    capital
    .set_index("pathway")
    .reindex(PATHWAY_ORDER)
)

fci_musd = capital_by_path["fci_usd"] / 1e6


# OPTIONAL:
# If you also want Total Installed Cost in Table S17,
# uncomment the following:
#
# installed_cost_musd = (
#     capital_by_path["total_installed_usd"] / 1e6
# )


# ------------------------------------------------------------
# 6. Annual OPEX
#
# Cash-flow signs:
# feed_cost, elec_cost, other_var_cost, fixed_cost are stored
# as negative cash flows.
#
# Thus positive OPEX is:
# -(feed + electricity + other variable + fixed)
# ------------------------------------------------------------
cashflows["annual_opex_usd"] = -(
    cashflows["feed_cost"]
    + cashflows["elec_cost"]
    + cashflows["other_var_cost"]
    + cashflows["fixed_cost"]
)


# ------------------------------------------------------------
# 7. Use normal, full-capacity operating years
#
# Year 1 contains the startup/ramp assumptions.
# Years >= 2 are full-capacity years and are constant in the
# current deterministic TEA.
#
# Taking the mean of years >= 2 makes this robust and gives
# the steady-state annual OPEX and byproduct revenue.
# ------------------------------------------------------------
steady_state = (
    cashflows.loc[cashflows["year"] >= 2]
    .groupby("pathway", as_index=True)
    .agg(
        annual_opex_usd=("annual_opex_usd", "mean"),
        annual_byproduct_credit_usd=("byprod_rev", "mean"),
    )
    .reindex(PATHWAY_ORDER)
)

opex_musd_yr = steady_state["annual_opex_usd"] / 1e6
byproduct_musd_yr = (
    steady_state["annual_byproduct_credit_usd"] / 1e6
)


# ------------------------------------------------------------
# 8. Construct revised Table S17
# ------------------------------------------------------------
table_s17 = pd.DataFrame({
    "Pathway": PATHWAY_ORDER,

    "Annual graphite amount\n(thousand tonne/yr)":
        annual_graphite_kt.values,

    "Fixed capital investment, FCI\n(million $)":
        fci_musd.values,

    "Annual OPEX\n(million $/yr)":
        opex_musd_yr.values,

    "Annual by-product credit\n(million $/yr)":
        byproduct_musd_yr.values,

    "Precursor throughput\n(thousand kg/h)":
        precursor_rate_thousand.values,
})


# ------------------------------------------------------------
# 9. Round for SI presentation
# ------------------------------------------------------------
table_s17_display = table_s17.copy()

table_s17_display[
    "Annual graphite amount\n(thousand tonne/yr)"
] = table_s17_display[
    "Annual graphite amount\n(thousand tonne/yr)"
].round(1)

table_s17_display[
    "Fixed capital investment, FCI\n(million $)"
] = table_s17_display[
    "Fixed capital investment, FCI\n(million $)"
].round(1)

table_s17_display[
    "Annual OPEX\n(million $/yr)"
] = table_s17_display[
    "Annual OPEX\n(million $/yr)"
].round(1)

table_s17_display[
    "Annual by-product credit\n(million $/yr)"
] = table_s17_display[
    "Annual by-product credit\n(million $/yr)"
].round(1)

table_s17_display[
    "Precursor throughput\n(thousand kg/h)"
] = table_s17_display[
    "Precursor throughput\n(thousand kg/h)"
].round(2)


# ------------------------------------------------------------
# 10. Basic consistency checks
# ------------------------------------------------------------

# Every pathway must appear once
assert table_s17["Pathway"].tolist() == PATHWAY_ORDER

# All production values must be positive
assert (
    table_s17[
        "Annual graphite amount\n(thousand tonne/yr)"
    ] > 0
).all()

# No negative OPEX
assert (
    table_s17[
        "Annual OPEX\n(million $/yr)"
    ] > 0
).all()

# BC1 and BO1 should have no credited coproduct revenue
assert np.isclose(
    table_s17.loc[
        table_s17["Pathway"].eq("BC1"),
        "Annual by-product credit\n(million $/yr)"
    ].iloc[0],
    0.0,
)

assert np.isclose(
    table_s17.loc[
        table_s17["Pathway"].eq("BO1"),
        "Annual by-product credit\n(million $/yr)"
    ].iloc[0],
    0.0,
)


# ------------------------------------------------------------
# 11. Display and export
# ------------------------------------------------------------
display(table_s17_display)

table_s17_display.to_csv(
    OUTPUT_DIR / "table_S17_revised.csv",
    index=False
)

table_s17_display.to_excel(
    OUTPUT_DIR / "table_S17_revised.xlsx",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "table_S17_revised.csv"
)
print(
    "Saved:",
    OUTPUT_DIR / "table_S17_revised.xlsx"
)

Operating hours used: 8,000 h/yr


,Pathway,Annual graphite amount\n(thousand tonne/yr),"Fixed capital investment, FCI\n(million $)",Annual OPEX\n(million $/yr),Annual by-product credit\n(million $/yr),Precursor throughput\n(thousand kg/h)
0,BC1,134.5,254.1,545.7,0.0,21.87
1,BC2,134.5,507.4,579.6,55.3,21.87
2,BO1,75.7,648.5,370.6,0.0,12.25
3,BO2,75.7,921.1,380.0,99.4,12.25


Saved: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/reproducibility/table_S17_revised.csv
Saved: /mnt/g/My Drive/yale/Project-graphite/graphite_clean_project_spatial_v2_with_03b_legacy_visualization/graphite_clean_project_spatial_v2/result/reproducibility/table_S17_revised.xlsx
